In [32]:
import numpy as np
from limvam.praline import estimate_causal_order

# generate data

In [33]:
# parameters
m = 5
p = 4
n = 1000
random_state = 42
rng = np.random.RandomState(random_state)
steps = 1000
lr = 1e-2

In [34]:
def generate_data(
    m,
    p,
    n,
    rng=None,
):
    # variance of the disturbances
    M = rng.randn(p, m, m)
    Sigmas = np.zeros((p, m, m))
    for j in range(p):
        Sigmas[j] = M[j] @ M[j].T + m * np.eye(m)
    
    # disturbances
    E = np.zeros((m, p, n))
    for j in range(p):
        E[:, j] = rng.multivariate_normal(mean=np.zeros(m), cov=Sigmas[j], size=(n,)).T

    # strictly lower triangular matrices T
    T = rng.normal(size=(m, p, p))
    for i in range(m):
        T[i][np.triu_indices(p, k=0)] = 0
    
    # causal order P
    order = rng.permutation(p)
    P = np.eye(p)[order]

    # causal effect matrices B
    B = P.T @ T @ P
    
    # mixing matrices
    A = np.linalg.inv(np.eye(p) - B)
    
    # observations
    X = np.array([Ai @ Ei for Ai, Ei in zip(A, E)])
    return X, B, T, P, order

In [35]:
X, _, _, _, order_true = generate_data(m, p, n, rng)

# PRaLiNE

In [36]:
order_estimate = estimate_causal_order(X, steps=steps, lr=lr)

In [37]:
print(f"True order : {order_true}")
print(f"Estimated order : {order_estimate}")

True order : [1 3 2 0]
Estimated order : [1, 3, 2, 0]
